# Recod.ai/LUC Scientific Image Forgery Detection - Full Data 4 Model Study

Bu notebook, Recod.ai/LUC scientific image forgery detection problemi icin ayni full-data shared split uzerinde
dort modeli adil ve tekrarlanabilir bicimde egitir, degerlendirir ve karsilastirir:

1. U-Net++ ResNet34 RGB
2. EfficientNetB0-UNet RGB
3. SegFormer-B0 RGB
4. DINOv2-lite decoder RGB

Notlar:
- Test set threshold veya post-processing secimi icin kullanilmaz.
- Pixel-level all ve forged-only metrikler ayri raporlanir.
- Component-aware metrikler egitim loss'u degil, final evaluation katmanidir.
- DINOv2 icin varsayilan model input boyutu 252, ortak evaluation boyutu 256'dir.

## 1. Install / Imports

In [ ]:
import os
import sys
import json
import math
import time
import random
import shutil
import platform
import warnings
import traceback
import subprocess
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", message=".*The secret `HF_TOKEN` does not exist.*")
warnings.filterwarnings("ignore", message=".*Some weights of.*were not initialized.*")

def is_kaggle_runtime() -> bool:
    return Path("/kaggle/input").exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

def is_colab_runtime() -> bool:
    return "google.colab" in sys.modules

def ensure_package(import_name: str, pip_name: Optional[str] = None) -> None:
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"[install] {pip_name} kuruluyor...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        except Exception as exc:
            raise RuntimeError(
                f"{pip_name} kurulamadı. Kaggle kullanıyorsan Internet'i açman veya paketi notebook environment'ına eklemen gerekir."
            ) from exc

ensure_package("cv2", "opencv-python-headless")
ensure_package("sklearn", "scikit-learn")
ensure_package("matplotlib")
ensure_package("tqdm")
ensure_package("albumentations")
ensure_package("segmentation_models_pytorch", "segmentation-models-pytorch")
ensure_package("transformers")
ensure_package("scipy")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    jaccard_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torch.amp import GradScaler, autocast
    TORCH_AMP_NEW_API = True
except Exception:
    from torch.cuda.amp import GradScaler, autocast
    TORCH_AMP_NEW_API = False

import albumentations as A
import segmentation_models_pytorch as smp
from transformers import AutoModel, SegformerForSemanticSegmentation

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Global Config

In [ ]:
KAGGLE_DATASET_CANDIDATES = [
    "/kaggle/input/competitions/recodai-luc-scientific-image-forgery-detection",
    "/kaggle/input/recodai-luc-scientific-image-forgery-detection",
    "/kaggle/input/datasets/koushikkumardinda/scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection",
    "/kaggle/input/scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection",
]
COLAB_DATASET_ROOT = "/content/drive/MyDrive/bitirmeProjesi/dataset"
COLAB_RUN_ROOT = "/content/drive/MyDrive/bitirmeProjesi/experiments_4_full"
COLAB_COMPARISON_ROOT = "/content/drive/MyDrive/bitirmeProjesi/experiments_full"
KAGGLE_RUN_ROOT = "/kaggle/working/experiments_4_full"
KAGGLE_COMPARISON_ROOT = "/kaggle/working/experiments_full"

def default_dataset_root() -> str:
    if is_kaggle_runtime():
        for candidate in KAGGLE_DATASET_CANDIDATES:
            if Path(candidate).exists():
                return candidate
        return KAGGLE_DATASET_CANDIDATES[0]
    return COLAB_DATASET_ROOT

def default_run_root() -> str:
    return KAGGLE_RUN_ROOT if is_kaggle_runtime() else COLAB_RUN_ROOT

def default_comparison_root() -> str:
    return KAGGLE_COMPARISON_ROOT if is_kaggle_runtime() else COLAB_COMPARISON_ROOT

@dataclass
class GlobalConfig:
    seed: int = 42
    dataset_root: str = default_dataset_root()
    run_root: str = default_run_root()
    comparison_root: str = default_comparison_root()
    shared_split_dir: str = "experiments/_shared_splits_seed42"

    image_size: int = 256
    dino_input_size: int = 252
    batch_size: int = 8
    dino_batch_size: int = 4
    num_workers: int = 0
    epochs: int = 40
    learning_rate: float = 1e-4
    backbone_learning_rate: float = 1e-5
    weight_decay: float = 1e-4
    scheduler_name: str = "reduce_on_plateau"
    early_stopping_patience: int = 8
    reduce_lr_patience: int = 3
    reduce_lr_factor: float = 0.5
    use_amp: bool = True
    use_pos_weight: bool = True
    pos_weight_min: float = 1.0
    pos_weight_max: float = 20.0
    pos_weight_scan_samples: int = 800
    gradient_accumulation_steps: int = 1

    pixel_thresholds: Tuple[float, ...] = tuple(np.round(np.arange(0.10, 0.901, 0.05), 2))
    image_thresholds: Tuple[float, ...] = tuple(np.round(np.arange(0.00, 1.001, 0.01), 2))
    min_component_areas: Tuple[int, ...] = (0, 25, 50, 100, 200, 500)
    min_component_mean_probs: Tuple[float, ...] = (0.0, 0.1, 0.2, 0.3)
    component_iou_thresholds: Tuple[float, ...] = (0.10, 0.25)
    primary_component_iou_threshold: float = 0.10
    image_score_methods: Tuple[str, ...] = ("max_probability", "pred_mask_ratio", "topk_mean_probability")
    topk_fraction: float = 0.01

    skip_existing: bool = True
    force_rerun: bool = False
    save_prediction_probs: bool = True
    max_prediction_examples: int = 12
    run_robustness: bool = False
    create_submission: bool = False
    run_all_experiments: bool = True

@dataclass
class ExperimentConfig:
    name: str
    model_type: str
    model_family: str
    encoder_or_backbone: str
    input_mode: str = "rgb"
    in_channels: int = 3
    classes: int = 1
    pretrained: Any = True
    encoder_weights: Optional[str] = "imagenet"
    hf_model_name: Optional[str] = None
    image_size: Optional[int] = None
    eval_size: Optional[int] = None
    batch_size: Optional[int] = None
    freeze_backbone_stage1: bool = False
    optional_unfreeze_last_blocks_stage2: bool = False
    skip_existing: bool = True
    run: bool = True

CFG = GlobalConfig()

EXPERIMENTS = [
    ExperimentConfig(
        name="unetpp_resnet34_rgb_full",
        model_type="unetplusplus",
        model_family="CNN encoder-decoder baseline",
        encoder_or_backbone="resnet34",
        encoder_weights="imagenet",
        skip_existing=True,
    ),
    ExperimentConfig(
        name="efficientnetb0_unet_rgb_full",
        model_type="unet",
        model_family="parameter-efficient CNN transfer baseline",
        encoder_or_backbone="efficientnet-b0",
        encoder_weights="imagenet",
    ),
    ExperimentConfig(
        name="segformer_b0_rgb_full",
        model_type="segformer",
        model_family="transformer semantic segmentation",
        encoder_or_backbone="nvidia/segformer-b0-finetuned-ade-512-512",
        hf_model_name="nvidia/segformer-b0-finetuned-ade-512-512",
        encoder_weights=None,
    ),
    ExperimentConfig(
        name="dinov2_lite_decoder_rgb_full",
        model_type="dinov2_lite_decoder",
        model_family="foundation-feature lightweight decoder",
        encoder_or_backbone="facebook/dinov2-small",
        hf_model_name="facebook/dinov2-small",
        encoder_weights=None,
        image_size=CFG.dino_input_size,
        eval_size=CFG.image_size,
        batch_size=CFG.dino_batch_size,
        freeze_backbone_stage1=True,
        optional_unfreeze_last_blocks_stage2=False,
    ),
]

# Hızlı smoke test istersen tek modeli acabilirsin:
# for exp in EXPERIMENTS:
#     exp.run = exp.name == "unetpp_resnet34_rgb_full"

print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False)[:2000])

## 3. Seed and Environment

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def runtime_name() -> str:
    if is_kaggle_runtime():
        return "kaggle"
    if is_colab_runtime():
        return "colab"
    return "local"

def mount_drive_if_colab() -> None:
    if is_colab_runtime():
        from google.colab import drive
        drive.mount("/content/drive")

def environment_info() -> Dict[str, Any]:
    info = {
        "runtime": runtime_name(),
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "device": str(DEVICE),
    }
    if torch.cuda.is_available():
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["gpu_memory_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    return info

seed_everything(CFG.seed)
mount_drive_if_colab()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Runtime:", runtime_name())
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Path Discovery

In [ ]:
DATASET_ROOT = Path(CFG.dataset_root)
RUN_ROOT = Path(CFG.run_root)
COMPARISON_ROOT = Path(CFG.comparison_root)
LOCAL_SHARED_SPLIT_DIR = Path(CFG.shared_split_dir)
SHARED_SPLIT_DIR = LOCAL_SHARED_SPLIT_DIR if LOCAL_SHARED_SPLIT_DIR.exists() else (RUN_ROOT.parent / "_shared_splits_seed42")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)
SHARED_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

AUTH_DIR = DATASET_ROOT / "train_images" / "authentic"
FORGED_DIR = DATASET_ROOT / "train_images" / "forged"
MASK_DIR = DATASET_ROOT / "train_masks"
TEST_IMAGE_DIR = DATASET_ROOT / "test_images"
SAMPLE_SUBMISSION = DATASET_ROOT / "sample_submission.csv"

print("Dataset root:", DATASET_ROOT)
print("Run root:", RUN_ROOT)
print("Comparison root:", COMPARISON_ROOT)
print("Shared split dir:", SHARED_SPLIT_DIR)
for path in [DATASET_ROOT, AUTH_DIR, FORGED_DIR, MASK_DIR, TEST_IMAGE_DIR, SAMPLE_SUBMISSION]:
    print(f"{path}: exists={path.exists()}")

## 5. Dataset Index Creation

In [ ]:
def image_id_from_path(path: Path) -> str:
    return path.stem

def find_mask_paths(image_id: str, mask_dir: Path) -> List[Path]:
    exact = mask_dir / f"{image_id}.npy"
    candidates = []
    if exact.exists():
        candidates.append(exact)
    # Birden fazla maske varsa aynı image_id ile başlayan dosyaları da birleştir.
    for path in sorted(mask_dir.glob(f"{image_id}*.npy")):
        if path not in candidates:
            candidates.append(path)
    return candidates

def build_dataset_index(dataset_root: Path) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    auth_paths = sorted((dataset_root / "train_images" / "authentic").glob("*.png"))
    forged_paths = sorted((dataset_root / "train_images" / "forged").glob("*.png"))
    mask_dir = dataset_root / "train_masks"

    for image_path in auth_paths:
        image_id = image_id_from_path(image_path)
        rows.append({
            "sample_id": f"authentic__{image_id}",
            "image_id": str(image_id),
            "class_name": "authentic",
            "image_label": 0,
            "label": 0,
            "image_path": str(image_path),
            "mask_paths": "",
            "mask_path": "",
        })

    missing_masks = []
    for image_path in forged_paths:
        image_id = image_id_from_path(image_path)
        masks = find_mask_paths(image_id, mask_dir)
        if not masks:
            missing_masks.append(str(image_path))
        rows.append({
            "sample_id": f"forged__{image_id}",
            "image_id": str(image_id),
            "class_name": "forged",
            "image_label": 1,
            "label": 1,
            "image_path": str(image_path),
            "mask_paths": "|".join(str(p) for p in masks),
            "mask_path": str(masks[0]) if masks else "",
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"Dataset bos gorunuyor. DATASET_ROOT dogru mu? {dataset_root}")
    if missing_masks:
        pd.DataFrame({"image_path": missing_masks}).to_csv(SHARED_SPLIT_DIR / "missing_forged_masks.csv", index=False)
        print(f"[warning] Maske bulunamayan forged goruntu sayisi: {len(missing_masks)}")
    return df.sort_values(["class_name", "image_id"]).reset_index(drop=True)

def repair_split_paths(df: pd.DataFrame, dataset_root: Path) -> pd.DataFrame:
    df = df.copy()
    if "image_label" not in df.columns and "label" in df.columns:
        df["image_label"] = df["label"].astype(int)
    if "label" not in df.columns:
        df["label"] = df["image_label"].astype(int)
    if "class_name" not in df.columns:
        df["class_name"] = np.where(df["image_label"].astype(int) == 1, "forged", "authentic")
    repaired_image_paths, repaired_mask_paths, first_mask_paths = [], [], []
    for row in df.itertuples(index=False):
        image_id = str(getattr(row, "image_id"))
        class_name = getattr(row, "class_name")
        image_path = dataset_root / "train_images" / class_name / f"{image_id}.png"
        masks = find_mask_paths(image_id, dataset_root / "train_masks") if class_name == "forged" else []
        repaired_image_paths.append(str(image_path))
        repaired_mask_paths.append("|".join(str(p) for p in masks))
        first_mask_paths.append(str(masks[0]) if masks else "")
    df["image_path"] = repaired_image_paths
    df["mask_paths"] = repaired_mask_paths
    df["mask_path"] = first_mask_paths
    df["image_id"] = df["image_id"].astype(str)
    df["image_label"] = df["image_label"].astype(int)
    df["label"] = df["image_label"].astype(int)
    return df

full_index_df = build_dataset_index(DATASET_ROOT)
print("Toplam satir:", len(full_index_df))
print(full_index_df["class_name"].value_counts())
full_index_df.head()

## 6. Shared Split Loading / Creation

In [ ]:
def split_files_exist(split_dir: Path) -> bool:
    return all((split_dir / name).exists() for name in ["full.csv", "train.csv", "val.csv", "test.csv"])

def group_level_frame(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("image_id", as_index=False)
        .agg(image_label=("image_label", "max"), n_rows=("sample_id", "count"))
        .sort_values("image_id")
        .reset_index(drop=True)
    )

def create_shared_split(df: pd.DataFrame, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    group_df = group_level_frame(df)
    groups = group_df["image_id"].to_numpy()
    y = group_df["image_label"].to_numpy()
    try:
        sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        trainval_idx, test_idx = next(sgkf_test.split(group_df, y, groups=groups))
    except Exception as exc:
        print("[warning] StratifiedGroupKFold test split basarisiz, GroupShuffleSplit kullaniliyor:", exc)
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
        trainval_idx, test_idx = next(splitter.split(group_df, y, groups=groups))

    trainval_groups = group_df.iloc[trainval_idx].reset_index(drop=True)
    test_groups = set(group_df.iloc[test_idx]["image_id"].astype(str))
    relative_val_fraction = 0.10 / 0.80
    n_splits_val = max(2, int(round(1 / relative_val_fraction)))
    try:
        sgkf_val = StratifiedGroupKFold(n_splits=n_splits_val, shuffle=True, random_state=seed + 1)
        train_idx, val_idx = next(
            sgkf_val.split(trainval_groups, trainval_groups["image_label"], groups=trainval_groups["image_id"])
        )
    except Exception as exc:
        print("[warning] StratifiedGroupKFold val split basarisiz, GroupShuffleSplit kullaniliyor:", exc)
        splitter = GroupShuffleSplit(n_splits=1, test_size=relative_val_fraction, random_state=seed + 1)
        train_idx, val_idx = next(
            splitter.split(trainval_groups, trainval_groups["image_label"], groups=trainval_groups["image_id"])
        )
    train_groups = set(trainval_groups.iloc[train_idx]["image_id"].astype(str))
    val_groups = set(trainval_groups.iloc[val_idx]["image_id"].astype(str))
    train_df = df[df["image_id"].isin(train_groups)].reset_index(drop=True)
    val_df = df[df["image_id"].isin(val_groups)].reset_index(drop=True)
    test_df = df[df["image_id"].isin(test_groups)].reset_index(drop=True)
    return train_df, val_df, test_df

def leakage_report(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> Dict[str, int]:
    train_ids, val_ids, test_ids = set(train_df["image_id"]), set(val_df["image_id"]), set(test_df["image_id"])
    report = {
        "train_val_image_id_overlap": len(train_ids & val_ids),
        "train_test_image_id_overlap": len(train_ids & test_ids),
        "val_test_image_id_overlap": len(val_ids & test_ids),
    }
    return report

def summarize_split(name: str, df: pd.DataFrame) -> Dict[str, Any]:
    counts = df["class_name"].value_counts().to_dict()
    summary = {
        "split": name,
        "total_rows": int(len(df)),
        "authentic": int(counts.get("authentic", 0)),
        "forged": int(counts.get("forged", 0)),
    }
    print(summary)
    return summary

if split_files_exist(SHARED_SPLIT_DIR):
    print(f"Shared split yukleniyor: {SHARED_SPLIT_DIR}")
    full_df = repair_split_paths(pd.read_csv(SHARED_SPLIT_DIR / "full.csv"), DATASET_ROOT)
    train_df = repair_split_paths(pd.read_csv(SHARED_SPLIT_DIR / "train.csv"), DATASET_ROOT)
    val_df = repair_split_paths(pd.read_csv(SHARED_SPLIT_DIR / "val.csv"), DATASET_ROOT)
    test_df = repair_split_paths(pd.read_csv(SHARED_SPLIT_DIR / "test.csv"), DATASET_ROOT)
else:
    print("Shared split bulunamadi, yeniden olusturuluyor.")
    full_df = full_index_df.copy()
    train_df, val_df, test_df = create_shared_split(full_df, CFG.seed)

for name, df_part in [("full", full_df), ("train", train_df), ("val", val_df), ("test", test_df)]:
    df_part.to_csv(SHARED_SPLIT_DIR / f"{name}.csv", index=False)

split_summary = {
    "full": summarize_split("full", full_df),
    "train": summarize_split("train", train_df),
    "val": summarize_split("val", val_df),
    "test": summarize_split("test", test_df),
    "leakage": leakage_report(train_df, val_df, test_df),
}
print("Leakage report:", split_summary["leakage"])
with open(SHARED_SPLIT_DIR / "split_summary.json", "w", encoding="utf-8") as f:
    json.dump(split_summary, f, indent=2, ensure_ascii=False)

assert all(v == 0 for v in split_summary["leakage"].values()), "Group leakage tespit edildi."

## 7. Mask Loading Utilities

In [ ]:
def load_rgb_image(path: str) -> np.ndarray:
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f"Goruntu okunamadi: {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

def binarize_loaded_mask(mask: np.ndarray) -> np.ndarray:
    if mask.ndim == 2:
        binary = mask > 0
    elif mask.ndim == 3:
        if mask.shape[0] <= 16 and mask.shape[1] > 16 and mask.shape[2] > 16:
            binary = np.any(mask > 0, axis=0)       # (C, H, W)
        else:
            binary = np.any(mask > 0, axis=-1)      # (H, W, C)
    else:
        raise ValueError(f"Desteklenmeyen maske shape: {mask.shape}")
    return binary.astype(np.float32)

def load_binary_mask(mask_paths: str, label: int, target_hw: Tuple[int, int]) -> np.ndarray:
    h, w = target_hw
    if int(label) == 0:
        return np.zeros((h, w), dtype=np.float32)
    paths = [p for p in str(mask_paths).split("|") if p and p != "nan"]
    if not paths:
        warnings.warn(f"Forged ornekte maske yok; sifir maske uretildi.")
        return np.zeros((h, w), dtype=np.float32)
    merged = None
    for path in paths:
        raw = np.load(path, allow_pickle=False)
        mask = binarize_loaded_mask(raw)
        if mask.shape != (h, w):
            mask = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(np.float32)
        merged = mask if merged is None else np.maximum(merged, mask)
    return (merged > 0).astype(np.float32)

def resize_mask_nearest(mask: np.ndarray, size: int) -> np.ndarray:
    return cv2.resize(mask.astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST).astype(np.float32)

def resize_prob_bilinear(prob: np.ndarray, size: int) -> np.ndarray:
    return cv2.resize(prob.astype(np.float32), (size, size), interpolation=cv2.INTER_LINEAR).astype(np.float32)

## 8. Dataset and Augmentations

In [ ]:
def make_image_compression_aug():
    try:
        return A.ImageCompression(quality_range=(70, 100), p=0.25)
    except TypeError:
        try:
            return A.ImageCompression(quality_lower=70, quality_upper=100, p=0.25)
        except TypeError:
            return A.JpegCompression(quality_lower=70, quality_upper=100, p=0.25)

def resize_aug(size: int):
    try:
        return A.Resize(size, size, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST)
    except TypeError:
        return A.Resize(size, size, interpolation=cv2.INTER_LINEAR)

def shift_scale_rotate_aug():
    kwargs = dict(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=15,
        interpolation=cv2.INTER_LINEAR,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.5,
    )
    try:
        return A.ShiftScaleRotate(**kwargs, mask_interpolation=cv2.INTER_NEAREST)
    except TypeError:
        return A.ShiftScaleRotate(**kwargs)

def get_train_transform(size: int) -> A.Compose:
    return A.Compose([
        resize_aug(size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        shift_scale_rotate_aug(),
        A.RandomBrightnessContrast(p=0.35),
        A.GaussianBlur(blur_limit=(3, 5), p=0.20),
        A.GaussNoise(p=0.20),
        make_image_compression_aug(),
    ])

def get_eval_transform(size: int) -> A.Compose:
    return A.Compose([
        resize_aug(size),
    ])

def normalize_image(image: np.ndarray) -> np.ndarray:
    image = image.astype(np.float32) / 255.0
    return (image - IMAGENET_MEAN) / IMAGENET_STD

class ForgeryDataset(Dataset):
    def __init__(self, df: pd.DataFrame, input_size: int, eval_size: int, split: str, augment: bool = False):
        self.df = df.reset_index(drop=True)
        self.input_size = int(input_size)
        self.eval_size = int(eval_size)
        self.split = split
        self.transform = get_train_transform(self.input_size) if augment else get_eval_transform(self.input_size)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        image = load_rgb_image(row["image_path"])
        h, w = image.shape[:2]
        mask = load_binary_mask(row.get("mask_paths", row.get("mask_path", "")), int(row["image_label"]), (h, w))
        transformed = self.transform(image=image, mask=mask)
        image_t = normalize_image(transformed["image"])
        mask_t = (transformed["mask"] > 0).astype(np.float32)
        eval_mask = resize_mask_nearest(mask_t, self.eval_size) if self.input_size != self.eval_size else mask_t
        return {
            "image": torch.from_numpy(image_t.transpose(2, 0, 1)).float(),
            "mask": torch.from_numpy(mask_t[None]).float(),
            "eval_mask": torch.from_numpy(eval_mask[None]).float(),
            "image_id": str(row["image_id"]),
            "image_path": str(row["image_path"]),
            "class_name": str(row["class_name"]),
            "image_label": int(row["image_label"]),
            "orig_h": int(h),
            "orig_w": int(w),
        }

def make_loaders(exp: ExperimentConfig) -> Tuple[DataLoader, DataLoader, DataLoader]:
    input_size = exp.image_size or CFG.image_size
    eval_size = exp.eval_size or CFG.image_size
    batch_size = exp.batch_size or CFG.batch_size
    train_ds = ForgeryDataset(train_df, input_size=input_size, eval_size=eval_size, split="train", augment=True)
    val_ds = ForgeryDataset(val_df, input_size=input_size, eval_size=eval_size, split="val", augment=False)
    test_ds = ForgeryDataset(test_df, input_size=input_size, eval_size=eval_size, split="test", augment=False)
    loader_kwargs = dict(batch_size=batch_size, num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_ds, shuffle=True, drop_last=False, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, drop_last=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, drop_last=False, **loader_kwargs)
    return train_loader, val_loader, test_loader

## 9. Model Builders

In [ ]:
class SegFormerBinaryWrapper(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            model_name,
            num_labels=1,
            ignore_mismatched_sizes=True,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.model(pixel_values=x)
        logits = out.logits
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return logits

class DINOv2LiteDecoder(nn.Module):
    def __init__(self, model_name: str = "facebook/dinov2-small", freeze_backbone: bool = True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = int(self.backbone.config.hidden_size)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.decoder = nn.Sequential(
            nn.Conv2d(hidden_size, 256, kernel_size=3, padding=1),
            nn.GroupNorm(16, 256),
            nn.GELU(),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.GroupNorm(8, 128),
            nn.GELU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, 1, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if all(not p.requires_grad for p in self.backbone.parameters()):
            with torch.no_grad():
                out = self.backbone(pixel_values=x)
        else:
            out = self.backbone(pixel_values=x)
        tokens = out.last_hidden_state[:, 1:, :]
        b, n, c = tokens.shape
        grid = int(math.sqrt(n))
        if grid * grid != n:
            tokens = tokens[:, : grid * grid, :]
        feat = tokens.transpose(1, 2).contiguous().view(b, c, grid, grid)
        logits = self.decoder(feat)
        logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return logits

def build_model(exp: ExperimentConfig) -> nn.Module:
    if exp.model_type == "unetplusplus":
        return smp.UnetPlusPlus(
            encoder_name=exp.encoder_or_backbone,
            encoder_weights=exp.encoder_weights,
            in_channels=exp.in_channels,
            classes=exp.classes,
            activation=None,
        )
    if exp.model_type == "unet":
        return smp.Unet(
            encoder_name=exp.encoder_or_backbone,
            encoder_weights=exp.encoder_weights,
            in_channels=exp.in_channels,
            classes=exp.classes,
            activation=None,
        )
    if exp.model_type == "segformer":
        return SegFormerBinaryWrapper(exp.hf_model_name or "nvidia/segformer-b0-finetuned-ade-512-512")
    if exp.model_type == "dinov2_lite_decoder":
        return DINOv2LiteDecoder(
            exp.hf_model_name or "facebook/dinov2-small",
            freeze_backbone=exp.freeze_backbone_stage1,
        )
    raise ValueError(f"Bilinmeyen model_type: {exp.model_type}")

def count_parameters(model: nn.Module) -> Tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

def save_model_summary(model: nn.Module, out_path: Path) -> None:
    trainable, total = count_parameters(model)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(str(model))
        f.write("\n\n")
        f.write(f"total_params={total}\ntrainable_params={trainable}\n")

## 10. Losses and Metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, eps: float = 1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)
        dims = (1, 2, 3)
        intersection = torch.sum(probs * targets, dims)
        union = torch.sum(probs, dims) + torch.sum(targets, dims)
        dice = (2.0 * intersection + self.eps) / (union + self.eps)
        return 1.0 - dice.mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, pos_weight: Optional[torch.Tensor] = None):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.dice = DiceLoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

def compute_positive_pixel_ratio(df: pd.DataFrame, image_size: int, max_samples: int) -> float:
    forged = df[df["image_label"].astype(int) == 1]
    if len(forged) == 0:
        return 0.0
    scan_df = forged.sample(n=min(max_samples, len(forged)), random_state=CFG.seed)
    pos, total = 0.0, 0.0
    for row in tqdm(scan_df.itertuples(index=False), total=len(scan_df), desc="pos pixel scan"):
        image = load_rgb_image(row.image_path)
        h, w = image.shape[:2]
        mask = load_binary_mask(getattr(row, "mask_paths", getattr(row, "mask_path", "")), int(row.image_label), (h, w))
        mask = resize_mask_nearest(mask, image_size)
        pos += float(mask.sum())
        total += float(mask.size)
    return pos / max(total, 1.0)

def make_loss(train_df: pd.DataFrame, image_size: int, device: torch.device) -> Tuple[nn.Module, float, float]:
    pos_ratio = compute_positive_pixel_ratio(train_df, image_size, CFG.pos_weight_scan_samples)
    if CFG.use_pos_weight and pos_ratio > 0:
        raw_pos_weight = (1.0 - pos_ratio) / max(pos_ratio, 1e-8)
        pos_weight_value = float(np.clip(raw_pos_weight, CFG.pos_weight_min, CFG.pos_weight_max))
        pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    else:
        pos_weight_value = 1.0
        pos_weight = None
    return BCEDiceLoss(pos_weight=pos_weight), float(pos_ratio), float(pos_weight_value)

def dice_iou_from_binary(pred: np.ndarray, gt: np.ndarray, eps: float = 1e-7) -> Tuple[float, float]:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    pred_sum = pred.sum()
    gt_sum = gt.sum()
    dice = (2 * inter + eps) / (pred_sum + gt_sum + eps)
    union = np.logical_or(pred, gt).sum()
    iou = (inter + eps) / (union + eps)
    return float(dice), float(iou)

def safe_auc(y_true: np.ndarray, y_score: np.ndarray, kind: str) -> float:
    try:
        if len(np.unique(y_true)) < 2:
            return float("nan")
        if kind == "auprc":
            return float(average_precision_score(y_true, y_score))
        return float(roc_auc_score(y_true, y_score))
    except Exception:
        return float("nan")

def pixel_metrics_from_records(records: List[Dict[str, Any]], threshold: float, clean: bool = False,
                               min_area: int = 0, min_mean_prob: float = 0.0,
                               forged_only: bool = False, compute_auc: bool = True) -> Dict[str, float]:
    tp = fp = tn = fn = 0
    pred_ratios, gt_ratios = [], []
    all_probs, all_gts = [], []
    for rec in records:
        if forged_only and int(rec["image_label"]) != 1:
            continue
        prob = rec["prob"]
        gt = rec["mask"].astype(np.uint8)
        pred = (prob >= threshold).astype(np.uint8)
        if clean:
            pred = clean_binary_mask(pred, prob, min_area=min_area, min_mean_prob=min_mean_prob)
        tp += int(((pred == 1) & (gt == 1)).sum())
        fp += int(((pred == 1) & (gt == 0)).sum())
        tn += int(((pred == 0) & (gt == 0)).sum())
        fn += int(((pred == 0) & (gt == 1)).sum())
        pred_ratios.append(float(pred.mean()))
        gt_ratios.append(float(gt.mean()))
        if compute_auc:
            all_probs.append(prob.reshape(-1).astype(np.float32))
            all_gts.append(gt.reshape(-1).astype(np.uint8))
    eps = 1e-7
    precision = tp / max(tp + fp, eps)
    recall = tp / max(tp + fn, eps)
    specificity = tn / max(tn + fp, eps)
    dice = (2 * tp) / max(2 * tp + fp + fn, eps)
    iou = tp / max(tp + fp + fn, eps)
    y_prob = np.concatenate(all_probs) if all_probs else np.array([])
    y_true = np.concatenate(all_gts) if all_gts else np.array([])
    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "auprc": safe_auc(y_true, y_prob, "auprc") if len(y_true) else float("nan"),
        "roc_auc": safe_auc(y_true, y_prob, "roc_auc") if len(y_true) else float("nan"),
        "pred_positive_pixel_ratio": float(np.mean(pred_ratios)) if pred_ratios else float("nan"),
        "gt_positive_pixel_ratio": float(np.mean(gt_ratios)) if gt_ratios else float("nan"),
    }

def image_score(prob: np.ndarray, pred: Optional[np.ndarray] = None, method: str = "topk_mean_probability") -> float:
    if method == "max_probability":
        return float(prob.max())
    if method == "pred_mask_ratio":
        if pred is None:
            pred = prob >= 0.5
        return float(pred.mean())
    if method == "topk_mean_probability":
        flat = prob.reshape(-1)
        k = max(1, int(len(flat) * CFG.topk_fraction))
        top = np.partition(flat, -k)[-k:]
        return float(top.mean())
    raise ValueError(method)

def image_level_metrics(records: List[Dict[str, Any]], pixel_threshold: float, image_threshold: float,
                        score_method: str, clean: bool = False, min_area: int = 0,
                        min_mean_prob: float = 0.0) -> Dict[str, float]:
    y_true, y_score = [], []
    for rec in records:
        prob = rec["prob"]
        pred = (prob >= pixel_threshold).astype(np.uint8)
        if clean:
            pred = clean_binary_mask(pred, prob, min_area=min_area, min_mean_prob=min_mean_prob)
        score = image_score(prob, pred, method=score_method)
        y_true.append(int(rec["image_label"]))
        y_score.append(score)
    y_true = np.array(y_true, dtype=np.uint8)
    y_score = np.array(y_score, dtype=np.float32)
    y_pred = (y_score >= image_threshold).astype(np.uint8)
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    return {
        "image_accuracy": float(accuracy_score(y_true, y_pred)),
        "image_precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "image_recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "image_specificity": float(tn / max(tn + fp, 1)),
        "image_f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "image_roc_auc": safe_auc(y_true, y_score, "roc_auc"),
        "image_score_method": score_method,
    }

## 11. Training Loop

In [ ]:
def autocast_context(device: torch.device, enabled: bool):
    enabled = bool(enabled and device.type == "cuda")
    if TORCH_AMP_NEW_API:
        return autocast(device_type=device.type, enabled=enabled)
    return autocast(enabled=enabled)

def forward_logits(model: nn.Module, images: torch.Tensor, target_size: Optional[Tuple[int, int]] = None) -> torch.Tensor:
    logits = model(images)
    if target_size is not None and logits.shape[-2:] != target_size:
        logits = F.interpolate(logits, size=target_size, mode="bilinear", align_corners=False)
    return logits

def train_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer,
                    scaler: GradScaler, device: torch.device, epoch: int) -> Dict[str, float]:
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f"train epoch {epoch}", leave=False)
    for step, batch in enumerate(pbar, start=1):
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        with autocast_context(device, CFG.use_amp):
            logits = forward_logits(model, images, target_size=masks.shape[-2:])
            loss = criterion(logits, masks) / CFG.gradient_accumulation_steps
        scaler.scale(loss).backward()
        if step % CFG.gradient_accumulation_steps == 0 or step == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running_loss += float(loss.detach().cpu()) * CFG.gradient_accumulation_steps
        pbar.set_postfix(loss=running_loss / step)
    return {"train_loss": running_loss / max(len(loader), 1)}

@torch.no_grad()
def validate_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Dict[str, float]:
    model.eval()
    losses = []
    tp = fp = fn = 0
    forged_tp = forged_fp = forged_fn = 0
    for batch in tqdm(loader, desc="val", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        with autocast_context(device, CFG.use_amp):
            logits = forward_logits(model, images, target_size=masks.shape[-2:])
            loss = criterion(logits, masks)
        probs = torch.sigmoid(logits).detach()
        preds = (probs >= 0.5).float()
        tp += int(((preds == 1) & (masks == 1)).sum().cpu())
        fp += int(((preds == 1) & (masks == 0)).sum().cpu())
        fn += int(((preds == 0) & (masks == 1)).sum().cpu())
        labels = batch["image_label"].numpy()
        for i, label in enumerate(labels):
            if int(label) == 1:
                p = preds[i:i+1]
                g = masks[i:i+1]
                forged_tp += int(((p == 1) & (g == 1)).sum().cpu())
                forged_fp += int(((p == 1) & (g == 0)).sum().cpu())
                forged_fn += int(((p == 0) & (g == 1)).sum().cpu())
        losses.append(float(loss.detach().cpu()))
    val_dice = (2 * tp) / max(2 * tp + fp + fn, 1e-7)
    val_forged_dice = (2 * forged_tp) / max(2 * forged_tp + forged_fp + forged_fn, 1e-7)
    return {
        "val_loss": float(np.mean(losses)) if losses else float("nan"),
        "val_dice": float(val_dice),
        "val_forged_dice": float(val_forged_dice),
    }

def make_optimizer(model: nn.Module, exp: ExperimentConfig) -> torch.optim.Optimizer:
    if exp.model_type == "dinov2_lite_decoder" and not exp.freeze_backbone_stage1:
        backbone_params, other_params = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if name.startswith("backbone."):
                backbone_params.append(p)
            else:
                other_params.append(p)
        return torch.optim.AdamW([
            {"params": other_params, "lr": CFG.learning_rate},
            {"params": backbone_params, "lr": CFG.backbone_learning_rate},
        ], weight_decay=CFG.weight_decay)
    return torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=CFG.learning_rate, weight_decay=CFG.weight_decay)

def make_scheduler(optimizer: torch.optim.Optimizer):
    if CFG.scheduler_name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=CFG.reduce_lr_factor, patience=CFG.reduce_lr_patience
    )

def checkpoint_state(model, optimizer, scheduler, epoch: int, best_score: float, exp: ExperimentConfig,
                     extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    return {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "epoch": int(epoch),
        "best_score": float(best_score),
        "experiment": asdict(exp),
        "global_config": asdict(CFG),
        "extra": extra or {},
    }

def load_checkpoint_if_available(path: Path, model, optimizer=None, scheduler=None, device: torch.device = DEVICE):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt and ckpt["optimizer_state_dict"] is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    return ckpt

def training_loop(exp: ExperimentConfig, out_dir: Path) -> Tuple[nn.Module, Dict[str, Any], pd.DataFrame]:
    train_loader, val_loader, _ = make_loaders(exp)
    model = build_model(exp).to(DEVICE)
    save_model_summary(model, out_dir / "model_summary.txt")
    trainable_params, total_params = count_parameters(model)
    criterion, pos_ratio, pos_weight = make_loss(train_df, exp.image_size or CFG.image_size, DEVICE)
    optimizer = make_optimizer(model, exp)
    scheduler = make_scheduler(optimizer)
    scaler = GradScaler(enabled=bool(CFG.use_amp and DEVICE.type == "cuda"))

    history: List[Dict[str, Any]] = []
    best_score = -1.0
    best_epoch = -1
    start_epoch = 1
    last_ckpt = out_dir / "last_model.pth"
    best_ckpt = out_dir / "best_model.pth"
    if last_ckpt.exists() and not CFG.force_rerun:
        ckpt = load_checkpoint_if_available(last_ckpt, model, optimizer, scheduler, DEVICE)
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_score = float(ckpt.get("best_score", -1.0))
        best_epoch = int(ckpt.get("extra", {}).get("best_epoch", -1))
        print(f"[resume] {exp.name}: epoch {start_epoch} itibariyle devam.")

    no_improve = 0
    for epoch in range(start_epoch, CFG.epochs + 1):
        t0 = time.time()
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE, epoch)
        val_metrics = validate_epoch(model, val_loader, criterion, DEVICE)
        score = val_metrics["val_forged_dice"] if not math.isnan(val_metrics["val_forged_dice"]) else val_metrics["val_dice"]
        if CFG.scheduler_name == "cosine":
            scheduler.step()
        else:
            scheduler.step(score)
        row = {
            "epoch": epoch,
            **train_metrics,
            **val_metrics,
            "lr": float(optimizer.param_groups[0]["lr"]),
            "epoch_time_minutes": (time.time() - t0) / 60.0,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(out_dir / "metrics.csv", index=False)
        improved = score > best_score
        if improved:
            best_score = float(score)
            best_epoch = int(epoch)
            no_improve = 0
            torch.save(checkpoint_state(
                model, optimizer, scheduler, epoch, best_score, exp,
                extra={"best_epoch": best_epoch, "pos_ratio": pos_ratio, "pos_weight": pos_weight}
            ), best_ckpt)
        else:
            no_improve += 1
        torch.save(checkpoint_state(
            model, optimizer, scheduler, epoch, best_score, exp,
            extra={"best_epoch": best_epoch, "pos_ratio": pos_ratio, "pos_weight": pos_weight}
        ), last_ckpt)
        print(f"{exp.name} epoch={epoch} loss={row['train_loss']:.4f} val_forged_dice={row['val_forged_dice']:.4f} best={best_score:.4f}")
        if no_improve >= CFG.early_stopping_patience:
            print(f"[early stopping] {exp.name}: {CFG.early_stopping_patience} epoch iyilesme yok.")
            break

    if best_ckpt.exists():
        load_checkpoint_if_available(best_ckpt, model, device=DEVICE)
    history_df = pd.DataFrame(history)
    meta = {
        "best_epoch": best_epoch,
        "best_val_forged_dice": best_score,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "pos_pixel_ratio": pos_ratio,
        "pos_weight": pos_weight,
    }
    return model, meta, history_df

def save_training_curves(metrics_df: pd.DataFrame, out_path: Path) -> None:
    if metrics_df.empty:
        return
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(metrics_df["epoch"], metrics_df["train_loss"], label="train")
    axes[0].plot(metrics_df["epoch"], metrics_df["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[1].plot(metrics_df["epoch"], metrics_df["val_dice"], label="all")
    axes[1].plot(metrics_df["epoch"], metrics_df["val_forged_dice"], label="forged")
    axes[1].set_title("Validation Dice")
    axes[1].legend()
    axes[2].plot(metrics_df["epoch"], metrics_df["lr"])
    axes[2].set_title("Learning rate")
    for ax in axes:
        ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)

## 12. Validation and Threshold Search

In [ ]:
@torch.no_grad()
def collect_predictions(model: nn.Module, loader: DataLoader, device: torch.device, desc: str) -> Tuple[List[Dict[str, Any]], float]:
    model.eval()
    records: List[Dict[str, Any]] = []
    n_images = 0
    t0 = time.time()
    for batch in tqdm(loader, desc=desc, leave=False):
        images = batch["image"].to(device, non_blocking=True)
        eval_masks = batch["eval_mask"].to(device, non_blocking=True)
        with autocast_context(device, CFG.use_amp):
            logits = forward_logits(model, images, target_size=eval_masks.shape[-2:])
        probs = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
        masks = eval_masks.detach().cpu().numpy()[:, 0]
        bs = probs.shape[0]
        for i in range(bs):
            records.append({
                "image_id": str(batch["image_id"][i]),
                "image_path": str(batch["image_path"][i]),
                "class_name": str(batch["class_name"][i]),
                "image_label": int(batch["image_label"][i]),
                "orig_h": int(batch["orig_h"][i]),
                "orig_w": int(batch["orig_w"][i]),
                "prob": probs[i].astype(np.float32),
                "mask": (masks[i] > 0).astype(np.uint8),
            })
        n_images += bs
    elapsed = time.time() - t0
    return records, 1000.0 * elapsed / max(n_images, 1)

def clean_binary_mask(binary: np.ndarray, prob: np.ndarray, min_area: int = 0, min_mean_prob: float = 0.0) -> np.ndarray:
    binary = binary.astype(np.uint8)
    if min_area <= 0 and min_mean_prob <= 0:
        return binary
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    cleaned = np.zeros_like(binary, dtype=np.uint8)
    for label_id in range(1, n):
        area = int(stats[label_id, cv2.CC_STAT_AREA])
        component = labels == label_id
        mean_prob = float(prob[component].mean()) if area > 0 else 0.0
        if area >= min_area and mean_prob >= min_mean_prob:
            cleaned[component] = 1
    return cleaned

def threshold_search_pixel(records: List[Dict[str, Any]]) -> Tuple[pd.DataFrame, Dict[str, Any], Dict[str, Any]]:
    rows = []
    for threshold in CFG.pixel_thresholds:
        m_all = pixel_metrics_from_records(records, threshold, clean=False, forged_only=False, compute_auc=False)
        m_forged = pixel_metrics_from_records(records, threshold, clean=False, forged_only=True, compute_auc=False)
        rows.append({
            "mode": "raw",
            "pixel_threshold": threshold,
            "min_component_area": 0,
            "min_component_mean_probability": 0.0,
            "dice_all": m_all["dice"],
            "dice_forged_only": m_forged["dice"],
            "iou_all": m_all["iou"],
            "iou_forged_only": m_forged["iou"],
        })
        for area in CFG.min_component_areas:
            for mean_prob in CFG.min_component_mean_probs:
                if area == 0 and mean_prob == 0:
                    continue
                mc_all = pixel_metrics_from_records(records, threshold, clean=True, min_area=area, min_mean_prob=mean_prob, forged_only=False, compute_auc=False)
                mc_forged = pixel_metrics_from_records(records, threshold, clean=True, min_area=area, min_mean_prob=mean_prob, forged_only=True, compute_auc=False)
                rows.append({
                    "mode": "clean",
                    "pixel_threshold": threshold,
                    "min_component_area": area,
                    "min_component_mean_probability": mean_prob,
                    "dice_all": mc_all["dice"],
                    "dice_forged_only": mc_forged["dice"],
                    "iou_all": mc_all["iou"],
                    "iou_forged_only": mc_forged["iou"],
                })
    df = pd.DataFrame(rows)
    raw_best = df[df["mode"] == "raw"].sort_values(["dice_forged_only", "iou_forged_only"], ascending=False).iloc[0].to_dict()
    clean_best = df[df["mode"] == "clean"].sort_values(["dice_forged_only", "iou_forged_only"], ascending=False).iloc[0].to_dict()
    return df, raw_best, clean_best

def threshold_search_image(records: List[Dict[str, Any]], selected_pixel_threshold: float,
                           clean: bool, min_area: int, min_mean_prob: float) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    rows = []
    for method in CFG.image_score_methods:
        scores, labels = [], []
        for rec in records:
            prob = rec["prob"]
            pred = (prob >= selected_pixel_threshold).astype(np.uint8)
            if clean:
                pred = clean_binary_mask(pred, prob, min_area=min_area, min_mean_prob=min_mean_prob)
            scores.append(image_score(prob, pred, method))
            labels.append(int(rec["image_label"]))
        scores = np.array(scores, dtype=np.float32)
        labels = np.array(labels, dtype=np.uint8)
        for thr in CFG.image_thresholds:
            pred_img = (scores >= thr).astype(np.uint8)
            tn = int(((pred_img == 0) & (labels == 0)).sum())
            fp = int(((pred_img == 1) & (labels == 0)).sum())
            rows.append({
                "score_method": method,
                "image_threshold": float(thr),
                "accuracy": float(accuracy_score(labels, pred_img)),
                "precision": float(precision_score(labels, pred_img, zero_division=0)),
                "recall": float(recall_score(labels, pred_img, zero_division=0)),
                "specificity": float(tn / max(tn + fp, 1)),
                "f1": float(f1_score(labels, pred_img, zero_division=0)),
                "roc_auc": safe_auc(labels, scores, "roc_auc"),
            })
    df = pd.DataFrame(rows)
    best = df.sort_values(["f1", "recall", "specificity"], ascending=False).iloc[0].to_dict()
    return df, best

def save_prediction_npz(records: List[Dict[str, Any]], out_path: Path) -> None:
    if not CFG.save_prediction_probs:
        return
    probs = np.stack([r["prob"].astype(np.float16) for r in records])
    masks = np.stack([r["mask"].astype(np.uint8) for r in records])
    image_ids = np.array([r["image_id"] for r in records])
    labels = np.array([r["image_label"] for r in records], dtype=np.uint8)
    np.savez_compressed(out_path, probs=probs, masks=masks, image_ids=image_ids, labels=labels)

## 13. Test Evaluation

In [ ]:
def per_image_metrics(records: List[Dict[str, Any]], pixel_threshold: float, image_threshold: float, score_method: str,
                      clean: bool = False, min_area: int = 0, min_mean_prob: float = 0.0) -> pd.DataFrame:
    rows = []
    for rec in records:
        prob = rec["prob"]
        gt = rec["mask"]
        pred = (prob >= pixel_threshold).astype(np.uint8)
        if clean:
            pred = clean_binary_mask(pred, prob, min_area=min_area, min_mean_prob=min_mean_prob)
        dice, iou = dice_iou_from_binary(pred, gt)
        score = image_score(prob, pred, method=score_method)
        rows.append({
            "image_id": rec["image_id"],
            "image_path": rec["image_path"],
            "class_name": rec["class_name"],
            "image_label": rec["image_label"],
            "gt_area": int(gt.sum()),
            "pred_area": int(pred.sum()),
            "dice": dice,
            "iou": iou,
            "image_score": score,
            "image_pred": int(score >= image_threshold),
            "gt_positive_pixel_ratio": float(gt.mean()),
            "pred_positive_pixel_ratio": float(pred.mean()),
        })
    return pd.DataFrame(rows)

def evaluate_records(records: List[Dict[str, Any]], selected: Dict[str, Any],
                     mode_name: str = "raw") -> Tuple[Dict[str, Any], pd.DataFrame]:
    clean = mode_name == "clean"
    pixel_threshold = float(selected["pixel_threshold"])
    min_area = int(selected.get("min_component_area", 0))
    min_mean_prob = float(selected.get("min_component_mean_probability", 0.0))
    image_threshold = float(selected["image_threshold"])
    score_method = str(selected["score_method"])

    all_m = pixel_metrics_from_records(records, pixel_threshold, clean=clean, min_area=min_area,
                                       min_mean_prob=min_mean_prob, forged_only=False)
    forged_m = pixel_metrics_from_records(records, pixel_threshold, clean=clean, min_area=min_area,
                                          min_mean_prob=min_mean_prob, forged_only=True)
    img_m = image_level_metrics(records, pixel_threshold, image_threshold, score_method,
                                clean=clean, min_area=min_area, min_mean_prob=min_mean_prob)
    per_df = per_image_metrics(records, pixel_threshold, image_threshold, score_method,
                              clean=clean, min_area=min_area, min_mean_prob=min_mean_prob)
    metrics = {
        "mode": mode_name,
        "selected_pixel_threshold": pixel_threshold,
        "selected_image_threshold": image_threshold,
        "score_method": score_method,
        "min_component_area": min_area,
        "min_component_mean_probability": min_mean_prob,
        "test_dice_all": all_m["dice"],
        "test_dice_forged_only": forged_m["dice"],
        "test_iou_all": all_m["iou"],
        "test_iou_forged_only": forged_m["iou"],
        "test_precision": all_m["precision"],
        "test_recall": all_m["recall"],
        "test_specificity": all_m["specificity"],
        "test_auprc_all": all_m["auprc"],
        "test_auprc_forged_only": forged_m["auprc"],
        "test_roc_auc_all": all_m["roc_auc"],
        "test_roc_auc_forged_only": forged_m["roc_auc"],
        "pred_positive_pixel_ratio": all_m["pred_positive_pixel_ratio"],
        "gt_positive_pixel_ratio": all_m["gt_positive_pixel_ratio"],
        **img_m,
    }
    return metrics, per_df

## 14. Component-aware Evaluation

In [ ]:
def component_table(binary: np.ndarray) -> Tuple[int, np.ndarray, np.ndarray]:
    binary = binary.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    return n, labels, stats

def component_iou_matrix(gt_labels: np.ndarray, gt_count: int, pred_labels: np.ndarray, pred_count: int) -> np.ndarray:
    if gt_count == 0 or pred_count == 0:
        return np.zeros((gt_count, pred_count), dtype=np.float32)
    mat = np.zeros((gt_count, pred_count), dtype=np.float32)
    for gi in range(1, gt_count + 1):
        gt_comp = gt_labels == gi
        for pi in range(1, pred_count + 1):
            pred_comp = pred_labels == pi
            inter = np.logical_and(gt_comp, pred_comp).sum()
            union = np.logical_or(gt_comp, pred_comp).sum()
            mat[gi - 1, pi - 1] = inter / max(union, 1)
    return mat

def match_components(gt: np.ndarray, pred: np.ndarray, iou_threshold: float) -> Dict[str, Any]:
    gt_n, gt_labels, _ = component_table(gt)
    pred_n, pred_labels, _ = component_table(pred)
    gt_count = gt_n - 1
    pred_count = pred_n - 1
    iou_mat = component_iou_matrix(gt_labels, gt_count, pred_labels, pred_count)
    matched = 0
    if gt_count > 0 and pred_count > 0:
        rows, cols = linear_sum_assignment(-iou_mat)
        for r, c in zip(rows, cols):
            if iou_mat[r, c] >= iou_threshold:
                matched += 1
    fp = pred_count - matched
    fn = gt_count - matched
    return {
        "gt_component_count": int(gt_count),
        "predicted_component_count": int(pred_count),
        "matched_component_count": int(matched),
        "false_positive_component_count": int(fp),
        "false_negative_component_count": int(fn),
    }

def component_metrics(records: List[Dict[str, Any]], selected: Dict[str, Any], mode_name: str = "clean",
                      iou_threshold: float = 0.10) -> Tuple[Dict[str, float], pd.DataFrame]:
    pixel_threshold = float(selected["pixel_threshold"])
    clean = mode_name == "clean"
    min_area = int(selected.get("min_component_area", 0))
    min_mean_prob = float(selected.get("min_component_mean_probability", 0.0))
    rows = []
    total_tp = total_fp = total_fn = 0
    authentic_with_pred = 0
    for rec in records:
        prob = rec["prob"]
        gt = rec["mask"].astype(np.uint8)
        pred = (prob >= pixel_threshold).astype(np.uint8)
        if clean:
            pred = clean_binary_mask(pred, prob, min_area=min_area, min_mean_prob=min_mean_prob)
        comp = match_components(gt, pred, iou_threshold=iou_threshold)
        total_tp += comp["matched_component_count"]
        total_fp += comp["false_positive_component_count"]
        total_fn += comp["false_negative_component_count"]
        auth_has_pred = bool(int(rec["image_label"]) == 0 and comp["predicted_component_count"] > 0)
        authentic_with_pred += int(auth_has_pred)
        rows.append({
            "image_id": rec["image_id"],
            "class_name": rec["class_name"],
            "image_label": rec["image_label"],
            "component_iou_threshold": iou_threshold,
            "authentic_image_has_prediction": auth_has_pred,
            **comp,
        })
    precision = total_tp / max(total_tp + total_fp, 1)
    recall = total_tp / max(total_tp + total_fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-7)
    authentic_n = sum(int(r["image_label"]) == 0 for r in records)
    metrics = {
        "component_iou_threshold": float(iou_threshold),
        "component_precision": float(precision),
        "component_recall": float(recall),
        "component_f1": float(f1),
        "authentic_fp_rate": float(authentic_with_pred / max(authentic_n, 1)),
        "avg_pred_component_count": float(np.mean([r["predicted_component_count"] for r in rows])) if rows else float("nan"),
    }
    return metrics, pd.DataFrame(rows)

## 15. Visualization and Failure Cases

In [ ]:
def make_overlay(image: np.ndarray, gt: np.ndarray, pred: np.ndarray) -> np.ndarray:
    overlay = image.copy().astype(np.float32)
    if overlay.max() <= 1.5:
        overlay = overlay * 255
    gt_bool = gt.astype(bool)
    pred_bool = pred.astype(bool)
    overlay[pred_bool] = 0.55 * overlay[pred_bool] + 0.45 * np.array([255, 0, 0])
    overlay[gt_bool] = 0.55 * overlay[gt_bool] + 0.45 * np.array([0, 255, 0])
    return np.clip(overlay, 0, 255).astype(np.uint8)

def make_error_map(gt: np.ndarray, pred: np.ndarray) -> np.ndarray:
    tp = (gt == 1) & (pred == 1)
    fp = (gt == 0) & (pred == 1)
    fn = (gt == 1) & (pred == 0)
    err = np.zeros((*gt.shape, 3), dtype=np.uint8)
    err[tp] = [0, 200, 0]
    err[fp] = [255, 0, 0]
    err[fn] = [0, 80, 255]
    return err

def denormalize_for_plot(image_path: str, size: int) -> np.ndarray:
    img = load_rgb_image(image_path)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)
    return img

def save_prediction_examples(records: List[Dict[str, Any]], selected: Dict[str, Any], out_path: Path, n: int = 12) -> None:
    if not records:
        return
    pixel_threshold = float(selected["pixel_threshold"])
    min_area = int(selected.get("min_component_area", 0))
    min_mean_prob = float(selected.get("min_component_mean_probability", 0.0))
    forged = [r for r in records if int(r["image_label"]) == 1]
    authentic = [r for r in records if int(r["image_label"]) == 0]
    chosen = (forged[: n // 2] + authentic[: n - n // 2]) if forged else records[:n]
    cols = ["image", "gt", "prob", "raw", "clean", "overlay", "error"]
    fig, axes = plt.subplots(len(chosen), len(cols), figsize=(2.5 * len(cols), 2.4 * len(chosen)))
    if len(chosen) == 1:
        axes = axes[None, :]
    for i, rec in enumerate(chosen):
        gt = rec["mask"].astype(np.uint8)
        prob = rec["prob"]
        raw = (prob >= pixel_threshold).astype(np.uint8)
        clean = clean_binary_mask(raw, prob, min_area=min_area, min_mean_prob=min_mean_prob)
        image = denormalize_for_plot(rec["image_path"], gt.shape[0])
        panels = [
            image,
            gt,
            prob,
            raw,
            clean,
            make_overlay(image, gt, clean),
            make_error_map(gt, clean),
        ]
        for j, panel in enumerate(panels):
            ax = axes[i, j]
            ax.imshow(panel, cmap="gray" if panel.ndim == 2 else None, vmin=0, vmax=1 if panel.ndim == 2 and panel.dtype != np.uint8 else None)
            ax.axis("off")
            if i == 0:
                ax.set_title(cols[j], fontsize=9)
        axes[i, 0].set_ylabel(f"{rec['class_name']}\n{rec['image_id']}", fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)

def add_component_counts_to_per_image(per_df: pd.DataFrame, comp_df: pd.DataFrame) -> pd.DataFrame:
    cols = ["image_id", "predicted_component_count", "gt_component_count"]
    if all(c in comp_df.columns for c in cols):
        return per_df.merge(comp_df[cols], on="image_id", how="left")
    return per_df

def save_failure_grid(records: List[Dict[str, Any]], failure_df: pd.DataFrame, selected: Dict[str, Any],
                      out_path: Path, title: str, n: int = 10) -> None:
    if failure_df.empty:
        return
    id_to_rec = {r["image_id"]: r for r in records}
    chosen_ids = failure_df["image_id"].head(n).astype(str).tolist()
    cols = ["image", "gt", "prob", "clean", "error"]
    fig, axes = plt.subplots(len(chosen_ids), len(cols), figsize=(2.7 * len(cols), 2.5 * len(chosen_ids)))
    if len(chosen_ids) == 1:
        axes = axes[None, :]
    pixel_threshold = float(selected["pixel_threshold"])
    min_area = int(selected.get("min_component_area", 0))
    min_mean_prob = float(selected.get("min_component_mean_probability", 0.0))
    for i, image_id in enumerate(chosen_ids):
        rec = id_to_rec[image_id]
        gt = rec["mask"].astype(np.uint8)
        prob = rec["prob"]
        raw = (prob >= pixel_threshold).astype(np.uint8)
        clean = clean_binary_mask(raw, prob, min_area=min_area, min_mean_prob=min_mean_prob)
        image = denormalize_for_plot(rec["image_path"], gt.shape[0])
        panels = [image, gt, prob, clean, make_error_map(gt, clean)]
        for j, panel in enumerate(panels):
            axes[i, j].imshow(panel, cmap="gray" if panel.ndim == 2 else None)
            axes[i, j].axis("off")
            if i == 0:
                axes[i, j].set_title(cols[j], fontsize=9)
        axes[i, 0].set_ylabel(str(image_id), fontsize=8)
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)

def save_failure_cases(records: List[Dict[str, Any]], per_df: pd.DataFrame, comp_df: pd.DataFrame,
                       selected: Dict[str, Any], out_dir: Path) -> None:
    enriched = add_component_counts_to_per_image(per_df, comp_df)
    low_dice = enriched[enriched["class_name"] == "forged"].sort_values("dice", ascending=True).head(10)
    fp_auth = enriched[(enriched["class_name"] == "authentic") & (enriched["pred_area"] > 0)].sort_values("pred_area", ascending=False).head(10)
    fn_forged = enriched[(enriched["class_name"] == "forged") & (enriched["pred_area"] <= 5)].sort_values("gt_area", ascending=False).head(10)
    low_dice.to_csv(out_dir / "failure_cases_low_dice_forged.csv", index=False)
    fp_auth.to_csv(out_dir / "failure_cases_false_positive_authentic.csv", index=False)
    fn_forged.to_csv(out_dir / "failure_cases_false_negative_forged.csv", index=False)
    save_failure_grid(records, low_dice, selected, out_dir / "failure_cases_low_dice_forged.png", "Low Dice Forged", n=10)
    save_failure_grid(records, fp_auth, selected, out_dir / "failure_cases_false_positive_authentic.png", "False Positive Authentic", n=10)
    save_failure_grid(records, fn_forged, selected, out_dir / "failure_cases_false_negative_forged.png", "False Negative Forged", n=10)

## 16. Model Comparison Table

In [ ]:
COMPARISON_COLUMNS = [
    "experiment_name", "model_family", "encoder_or_backbone", "input_mode", "image_size",
    "trainable_params", "total_params", "best_epoch", "selected_pixel_threshold",
    "selected_image_threshold", "min_component_area", "test_dice_all", "test_dice_forged_only",
    "test_iou_all", "test_iou_forged_only", "test_precision", "test_recall", "test_specificity",
    "test_auprc_all", "test_auprc_forged_only", "image_accuracy", "image_precision",
    "image_recall", "image_specificity", "image_f1", "image_roc_auc", "component_precision",
    "component_recall", "component_f1", "authentic_fp_rate", "avg_pred_component_count",
    "training_time_minutes", "inference_time_per_image_ms",
]

def load_existing_summary(exp: ExperimentConfig, out_dir: Path) -> Optional[Dict[str, Any]]:
    summary_path = out_dir / "summary.json"
    required = [out_dir / "best_model.pth", out_dir / "test_metrics.csv", out_dir / "test_per_image_metrics.csv"]
    if summary_path.exists() and all(p.exists() for p in required):
        with open(summary_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

def summary_to_comparison_row(summary: Dict[str, Any]) -> Dict[str, Any]:
    row = {k: summary.get(k, np.nan) for k in COMPARISON_COLUMNS}
    return row

def write_comparison_table() -> pd.DataFrame:
    rows = []
    for exp in EXPERIMENTS:
        summary_path = RUN_ROOT / exp.name / "summary.json"
        if summary_path.exists():
            with open(summary_path, "r", encoding="utf-8") as f:
                rows.append(summary_to_comparison_row(json.load(f)))
    comp = pd.DataFrame(rows)
    if comp.empty:
        print("Comparison icin henuz summary bulunamadi.")
        return comp
    sort_cols = ["test_dice_forged_only", "test_iou_forged_only", "component_f1", "authentic_fp_rate"]
    comp = comp.sort_values(sort_cols, ascending=[False, False, False, True]).reset_index(drop=True)
    for col in COMPARISON_COLUMNS:
        if col not in comp.columns:
            comp[col] = np.nan
    comp = comp[COMPARISON_COLUMNS]
    comp.to_csv(COMPARISON_ROOT / "model_comparison_full.csv", index=False)
    with open(COMPARISON_ROOT / "model_comparison_full.md", "w", encoding="utf-8") as f:
        f.write("# Full-data Model Comparison\n\n")
        f.write(comp.to_markdown(index=False))
        f.write("\n")
    return comp

## 17. Statistical Tests

In [ ]:
def run_statistical_tests() -> pd.DataFrame:
    from scipy.stats import ttest_rel, wilcoxon
    pairs = [
        ("unetpp_resnet34_rgb_full", "efficientnetb0_unet_rgb_full"),
        ("unetpp_resnet34_rgb_full", "segformer_b0_rgb_full"),
        ("unetpp_resnet34_rgb_full", "dinov2_lite_decoder_rgb_full"),
        ("segformer_b0_rgb_full", "dinov2_lite_decoder_rgb_full"),
    ]
    rows = []
    per_image = {}
    for exp in EXPERIMENTS:
        path = RUN_ROOT / exp.name / "test_per_image_metrics.csv"
        if path.exists():
            df = pd.read_csv(path)[["image_id", "dice", "iou"]].rename(columns={"dice": f"{exp.name}_dice", "iou": f"{exp.name}_iou"})
            df["image_id"] = df["image_id"].astype(str)
            per_image[exp.name] = df
    for a, b in pairs:
        if a not in per_image or b not in per_image:
            continue
        merged = per_image[a].merge(per_image[b], on="image_id", how="inner")
        if merged.empty:
            continue
        for metric in ["dice", "iou"]:
            x = merged[f"{a}_{metric}"].to_numpy(dtype=float)
            y = merged[f"{b}_{metric}"].to_numpy(dtype=float)
            diff = x - y
            try:
                t_stat, t_p = ttest_rel(x, y, nan_policy="omit")
            except Exception:
                t_stat, t_p = np.nan, np.nan
            try:
                w_stat, w_p = wilcoxon(diff)
            except Exception:
                w_stat, w_p = np.nan, np.nan
            rows.append({
                "model_a": a,
                "model_b": b,
                "metric": metric,
                "n_images": int(len(merged)),
                "mean_a": float(np.nanmean(x)),
                "mean_b": float(np.nanmean(y)),
                "mean_diff_a_minus_b": float(np.nanmean(diff)),
                "paired_t_stat": float(t_stat) if not np.isnan(t_stat) else np.nan,
                "paired_t_pvalue": float(t_p) if not np.isnan(t_p) else np.nan,
                "wilcoxon_stat": float(w_stat) if not np.isnan(w_stat) else np.nan,
                "wilcoxon_pvalue": float(w_p) if not np.isnan(w_p) else np.nan,
            })
    stats_df = pd.DataFrame(rows)
    stats_df.to_csv(COMPARISON_ROOT / "statistical_tests.csv", index=False)
    return stats_df

## 18. Optional Robustness

In [ ]:
def corrupt_image(image: np.ndarray, corruption: str) -> np.ndarray:
    if corruption == "jpeg90":
        _, enc = cv2.imencode(".jpg", cv2.cvtColor(image, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 90])
        return cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    if corruption == "jpeg70":
        _, enc = cv2.imencode(".jpg", cv2.cvtColor(image, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 70])
        return cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    if corruption == "gaussian_blur":
        return cv2.GaussianBlur(image, (5, 5), 0)
    if corruption == "gaussian_noise":
        noise = np.random.normal(0, 8, image.shape).astype(np.float32)
        return np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    raise ValueError(corruption)

class RobustnessDataset(ForgeryDataset):
    def __init__(self, df: pd.DataFrame, input_size: int, eval_size: int, corruption: str):
        super().__init__(df, input_size=input_size, eval_size=eval_size, split="test", augment=False)
        self.corruption = corruption

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        image = load_rgb_image(row["image_path"])
        image = corrupt_image(image, self.corruption)
        h, w = image.shape[:2]
        mask = load_binary_mask(row.get("mask_paths", row.get("mask_path", "")), int(row["image_label"]), (h, w))
        transformed = self.transform(image=image, mask=mask)
        image_t = normalize_image(transformed["image"])
        mask_t = (transformed["mask"] > 0).astype(np.float32)
        eval_mask = resize_mask_nearest(mask_t, self.eval_size) if self.input_size != self.eval_size else mask_t
        return {
            "image": torch.from_numpy(image_t.transpose(2, 0, 1)).float(),
            "mask": torch.from_numpy(mask_t[None]).float(),
            "eval_mask": torch.from_numpy(eval_mask[None]).float(),
            "image_id": str(row["image_id"]),
            "image_path": str(row["image_path"]),
            "class_name": str(row["class_name"]),
            "image_label": int(row["image_label"]),
            "orig_h": int(h),
            "orig_w": int(w),
        }

def run_robustness_for_best(best_exp_name: str) -> Optional[pd.DataFrame]:
    if not CFG.run_robustness:
        return None
    exp = next(e for e in EXPERIMENTS if e.name == best_exp_name)
    out_dir = RUN_ROOT / exp.name
    summary_path = out_dir / "summary.json"
    if not summary_path.exists():
        return None
    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)
    selected = {
        "pixel_threshold": summary["selected_pixel_threshold"],
        "image_threshold": summary["selected_image_threshold"],
        "score_method": summary.get("score_method", "topk_mean_probability"),
        "min_component_area": summary.get("min_component_area", 0),
        "min_component_mean_probability": summary.get("min_component_mean_probability", 0.0),
    }
    model = build_model(exp).to(DEVICE)
    load_checkpoint_if_available(out_dir / "best_model.pth", model, device=DEVICE)
    rows = []
    for corruption in ["jpeg90", "jpeg70", "gaussian_blur", "gaussian_noise"]:
        ds = RobustnessDataset(test_df, exp.image_size or CFG.image_size, exp.eval_size or CFG.image_size, corruption)
        loader = DataLoader(ds, batch_size=exp.batch_size or CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
        records, _ = collect_predictions(model, loader, DEVICE, desc=f"robustness {corruption}")
        metrics, _ = evaluate_records(records, selected, mode_name="clean")
        comp, _ = component_metrics(records, selected, mode_name="clean", iou_threshold=CFG.primary_component_iou_threshold)
        rows.append({
            "corruption": corruption,
            "dice": metrics["test_dice_forged_only"],
            "iou": metrics["test_iou_forged_only"],
            "auprc": metrics["test_auprc_forged_only"],
            "image_f1": metrics["image_f1"],
            "component_f1": comp["component_f1"],
        })
    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "robustness_metrics.csv", index=False)
    return df

## 19. Optional Kaggle Submission

In [ ]:
def mask_to_rle(mask: np.ndarray) -> str:
    pixels = mask.astype(np.uint8).T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

def create_optional_submission(best_exp_name: str) -> Optional[pd.DataFrame]:
    if not CFG.create_submission:
        return None
    if not SAMPLE_SUBMISSION.exists():
        print("[submission] sample_submission.csv bulunamadi.")
        return None
    print("[submission] Format kesin olmayabilir; dosya sample_submission kolonlarina gore doldurulacak.")
    exp = next(e for e in EXPERIMENTS if e.name == best_exp_name)
    out_dir = RUN_ROOT / exp.name
    with open(out_dir / "summary.json", "r", encoding="utf-8") as f:
        summary = json.load(f)
    selected = {
        "pixel_threshold": summary["selected_pixel_threshold"],
        "min_component_area": summary.get("min_component_area", 0),
        "min_component_mean_probability": summary.get("min_component_mean_probability", 0.0),
    }
    model = build_model(exp).to(DEVICE)
    load_checkpoint_if_available(out_dir / "best_model.pth", model, device=DEVICE)
    model.eval()
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    id_col = sample.columns[0]
    pred_col = sample.columns[-1]
    preds = {}
    transform = get_eval_transform(exp.image_size or CFG.image_size)
    for img_path in tqdm(sorted(TEST_IMAGE_DIR.glob("*.png")), desc="submission"):
        image = load_rgb_image(str(img_path))
        h, w = image.shape[:2]
        dummy_mask = np.zeros((h, w), dtype=np.float32)
        tr = transform(image=image, mask=dummy_mask)
        img_t = torch.from_numpy(normalize_image(tr["image"]).transpose(2, 0, 1))[None].float().to(DEVICE)
        with torch.no_grad(), autocast_context(DEVICE, CFG.use_amp):
            logits = forward_logits(model, img_t, target_size=(exp.image_size or CFG.image_size, exp.image_size or CFG.image_size))
            prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
        binary = (prob >= float(selected["pixel_threshold"])).astype(np.uint8)
        binary = clean_binary_mask(binary, prob, int(selected["min_component_area"]), float(selected["min_component_mean_probability"]))
        preds[img_path.stem] = mask_to_rle(binary)
    sample[pred_col] = sample[id_col].astype(str).map(preds).fillna("")
    out_path = out_dir / "submission.csv"
    sample.to_csv(out_path, index=False)
    print("Submission kaydedildi:", out_path)
    return sample

## 20. Auto Report Generation

In [ ]:
def write_experiment_report(exp: ExperimentConfig, summary: Dict[str, Any], out_dir: Path) -> None:
    lines = [
        f"# {exp.name}",
        "",
        "## Deney Tanimi",
        f"- Model ailesi: {exp.model_family}",
        f"- Encoder/backbone: {exp.encoder_or_backbone}",
        f"- Input mode: {exp.input_mode}",
        f"- Image size: {summary.get('image_size')}",
        f"- Trainable params: {summary.get('trainable_params')}",
        f"- Total params: {summary.get('total_params')}",
        "",
        "## Secilen Validation Ayarlari",
        f"- Pixel threshold: {summary.get('selected_pixel_threshold')}",
        f"- Image threshold: {summary.get('selected_image_threshold')}",
        f"- Image score: {summary.get('score_method')}",
        f"- Min component area: {summary.get('min_component_area')}",
        "",
        "## Test Sonuclari",
        f"- Forged-only Dice: {summary.get('test_dice_forged_only')}",
        f"- Forged-only IoU: {summary.get('test_iou_forged_only')}",
        f"- All-image Dice: {summary.get('test_dice_all')}",
        f"- AUPRC forged-only: {summary.get('test_auprc_forged_only')}",
        f"- Image F1: {summary.get('image_f1')}",
        f"- Component F1: {summary.get('component_f1')}",
        f"- Authentic FP rate: {summary.get('authentic_fp_rate')}",
        "",
        "## Yorum",
        "Bu otomatik rapor sayisal ciktinin ozetidir. Nihai bitirme projesi yorumu icin modeller arasi comparison ve failure-case gorselleri birlikte incelenmelidir.",
    ]
    with open(out_dir / "report.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

def write_global_report(comp: pd.DataFrame) -> None:
    report_path = COMPARISON_ROOT / "report.md"
    lines = ["# Full-data Dort Model Karsilastirma Raporu", ""]
    if comp.empty:
        lines.append("Henuz tamamlanmis deney bulunmuyor.")
    else:
        best = comp.iloc[0]
        lines += [
            "## Ana Bulgular",
            f"1. Full-data uzerinde siralama kriterlerine gore en guclu model: **{best['experiment_name']}**.",
            "2. U-Net++ ResNet34 RGB baseline referans model olarak comparison tablosunda ayrica izlenmelidir.",
            "3. EfficientNetB0-UNet parametre-verimli CNN baseline olarak Dice/parametre dengesini gosterir.",
            "4. SegFormer-B0 transformer tabanli modelin pilot basarisinin full-data'da korunup korunmadigi forged-only Dice ve component F1 ile okunmalidir.",
            "5. DINOv2-lite foundation feature yaklasiminin avantaji, ozellikle component F1 ve authentic FP rate tarafinda degerlendirilmelidir.",
            "",
            "## Comparison Table",
            comp.to_markdown(index=False),
            "",
            "## Degerlendirme Notu",
            "All-image pixel metric authentic sifir maskeler nedeniyle tek basina ana basari kriteri degildir. Forged-only Dice/IoU, component F1 ve authentic FP rate birlikte raporlanmalidir.",
        ]
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

def run_experiment(exp: ExperimentConfig) -> Optional[Dict[str, Any]]:
    if not exp.run:
        print(f"[skip] {exp.name}: run=False")
        return None
    out_dir = RUN_ROOT / exp.name
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = load_existing_summary(exp, out_dir)
    if existing is not None and exp.skip_existing and CFG.skip_existing and not CFG.force_rerun:
        print(f"[skip_existing] {exp.name}: mevcut sonuc comparison'a dahil edilecek.")
        return existing

    with open(out_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump({"global_config": asdict(CFG), "experiment": asdict(exp)}, f, indent=2, ensure_ascii=False)
    with open(out_dir / "environment_info.json", "w", encoding="utf-8") as f:
        json.dump(environment_info(), f, indent=2, ensure_ascii=False)

    t0_train = time.time()
    model, train_meta, history_df = training_loop(exp, out_dir)
    training_time_minutes = (time.time() - t0_train) / 60.0
    save_training_curves(history_df, out_dir / "training_curves.png")

    _, val_loader, test_loader = make_loaders(exp)
    val_records, val_infer_ms = collect_predictions(model, val_loader, DEVICE, desc=f"{exp.name} val predict")
    save_prediction_npz(val_records, out_dir / "val_predictions_probs.npz")
    pixel_search_df, raw_best, clean_best = threshold_search_pixel(val_records)
    pixel_search_df.to_csv(out_dir / "threshold_search_pixel.csv", index=False)

    raw_img_df, raw_img_best = threshold_search_image(
        val_records, float(raw_best["pixel_threshold"]), clean=False, min_area=0, min_mean_prob=0.0
    )
    clean_img_df, clean_img_best = threshold_search_image(
        val_records, float(clean_best["pixel_threshold"]), clean=True,
        min_area=int(clean_best["min_component_area"]),
        min_mean_prob=float(clean_best["min_component_mean_probability"]),
    )
    image_search_df = pd.concat([
        raw_img_df.assign(mode="raw"),
        clean_img_df.assign(mode="clean"),
    ], ignore_index=True)
    image_search_df.to_csv(out_dir / "threshold_search_image.csv", index=False)

    raw_selected = {**raw_best, **raw_img_best}
    clean_selected = {**clean_best, **clean_img_best}

    val_raw_metrics, _ = evaluate_records(val_records, raw_selected, mode_name="raw")
    val_clean_metrics, _ = evaluate_records(val_records, clean_selected, mode_name="clean")
    pd.DataFrame([val_raw_metrics, val_clean_metrics]).to_csv(out_dir / "val_metrics.csv", index=False)

    test_records, test_infer_ms = collect_predictions(model, test_loader, DEVICE, desc=f"{exp.name} test predict")
    save_prediction_npz(test_records, out_dir / "test_predictions_probs.npz")
    raw_metrics, raw_per_df = evaluate_records(test_records, raw_selected, mode_name="raw")
    clean_metrics, clean_per_df = evaluate_records(test_records, clean_selected, mode_name="clean")
    raw_comp_rows = []
    clean_comp_rows = []
    primary_comp_metrics = {}
    primary_comp_df = pd.DataFrame()
    for iou_thr in CFG.component_iou_thresholds:
        cm, cdf = component_metrics(test_records, clean_selected, mode_name="clean", iou_threshold=iou_thr)
        clean_comp_rows.append(cm)
        cdf["mode"] = "clean"
        if float(iou_thr) == float(CFG.primary_component_iou_threshold):
            primary_comp_metrics = cm
            primary_comp_df = cdf
        rm, rdf = component_metrics(test_records, raw_selected, mode_name="raw", iou_threshold=iou_thr)
        raw_comp_rows.append(rm)

    pd.DataFrame([raw_metrics]).to_csv(out_dir / "test_metrics_raw.csv", index=False)
    pd.DataFrame([clean_metrics]).to_csv(out_dir / "test_metrics_clean.csv", index=False)
    pd.DataFrame([clean_metrics]).to_csv(out_dir / "test_metrics.csv", index=False)
    clean_per_df.to_csv(out_dir / "test_per_image_metrics.csv", index=False)
    primary_comp_df.to_csv(out_dir / "test_component_metrics.csv", index=False)
    pd.DataFrame(clean_comp_rows).to_csv(out_dir / "test_component_summary_by_iou.csv", index=False)

    save_prediction_examples(test_records, clean_selected, out_dir / "prediction_examples.png", n=CFG.max_prediction_examples)
    save_failure_cases(test_records, clean_per_df, primary_comp_df, clean_selected, out_dir)

    summary = {
        "experiment_name": exp.name,
        "model_family": exp.model_family,
        "encoder_or_backbone": exp.encoder_or_backbone,
        "input_mode": exp.input_mode,
        "image_size": exp.image_size or CFG.image_size,
        "eval_size": exp.eval_size or CFG.image_size,
        "best_epoch": train_meta.get("best_epoch"),
        "trainable_params": train_meta.get("trainable_params"),
        "total_params": train_meta.get("total_params"),
        "pos_pixel_ratio": train_meta.get("pos_pixel_ratio"),
        "pos_weight": train_meta.get("pos_weight"),
        "selected_pixel_threshold": clean_metrics["selected_pixel_threshold"],
        "selected_image_threshold": clean_metrics["selected_image_threshold"],
        "score_method": clean_metrics["score_method"],
        "min_component_area": clean_metrics["min_component_area"],
        "min_component_mean_probability": clean_metrics["min_component_mean_probability"],
        "training_time_minutes": training_time_minutes,
        "inference_time_per_image_ms": test_infer_ms,
        **clean_metrics,
        **primary_comp_metrics,
    }
    with open(out_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    write_experiment_report(exp, summary, out_dir)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

def run_all_experiments() -> pd.DataFrame:
    error_log = RUN_ROOT / "experiment_errors.log"
    summaries = []
    for exp in EXPERIMENTS:
        try:
            summary = run_experiment(exp)
            if summary is not None:
                summaries.append(summary)
        except Exception as exc:
            print(f"[ERROR] {exp.name}: {exc}")
            with open(error_log, "a", encoding="utf-8") as f:
                f.write(f"\n\n===== {exp.name} =====\n")
                f.write(traceback.format_exc())
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue
    comp = write_comparison_table()
    stats = run_statistical_tests()
    write_global_report(comp)
    if not comp.empty:
        run_robustness_for_best(str(comp.iloc[0]["experiment_name"]))
        create_optional_submission(str(comp.iloc[0]["experiment_name"]))
    return comp

if CFG.run_all_experiments:
    comparison_df = run_all_experiments()
    display(comparison_df)
else:
    print("CFG.run_all_experiments=False. Tek model icin run_experiment(EXPERIMENTS[i]) cagirabilirsin.")